# [Module 04 -- Fundamentals] Built-in Tools

> **MLCourse -- Agentic AI -- CrewAI Fundamentals**

> CrewAI ships with a library of ready-made tools via the `crewai[tools]`
> package. This module covers the most useful built-in tools: file I/O,
> directory operations, web scraping, and search. We also show how to assign
> tools to agents and how to run tools standalone with `tool.run()`.

## What you'll learn

- `FileReadTool` -- read file contents.
- `FileWriterTool` -- write content to files.
- `DirectoryReadTool` -- list directory contents.
- `DirectorySearchTool` -- search for files by pattern.
- `ScrapeWebsiteTool` -- scrape web page content.
- `SerperDevTool` -- web search via Serper API (requires API key).
- How to assign tools to agents.
- Running tools standalone with `tool.run()`.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os
import tempfile
from pathlib import Path

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv

# Walk up to track root.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root:", TRACK)

## 1. Imports and crewai_tools verification

In [ ]:
try:
    from crewai import Agent, Task, Crew, Process
    from langchain_ollama import ChatOllama
    print("[OK] crewai and ChatOllama imported.")
except ImportError as e:
    print("[ERROR] pip install crewai \"crewai[tools]\" langchain-ollama")
    print("        Detail:", e)

In [ ]:
# Import the built-in tools. Each import is guarded so the notebook stays
# runnable even if crewai[tools] is not installed.
try:
    from crewai_tools import FileReadTool
    print("[OK] FileReadTool imported.")
except ImportError:
    print("[WARN] FileReadTool not available. Run: pip install \"crewai[tools]\"")
    FileReadTool = None

try:
    from crewai_tools import FileWriterTool
    print("[OK] FileWriterTool imported.")
except ImportError:
    print("[WARN] FileWriterTool not available.")
    FileWriterTool = None

try:
    from crewai_tools import DirectoryReadTool
    print("[OK] DirectoryReadTool imported.")
except ImportError:
    print("[WARN] DirectoryReadTool not available.")
    DirectoryReadTool = None

try:
    from crewai_tools import DirectorySearchTool
    print("[OK] DirectorySearchTool imported.")
except ImportError:
    print("[WARN] DirectorySearchTool not available.")
    DirectorySearchTool = None

try:
    from crewai_tools import ScrapeWebsiteTool
    print("[OK] ScrapeWebsiteTool imported.")
except ImportError:
    print("[WARN] ScrapeWebsiteTool not available.")
    ScrapeWebsiteTool = None

try:
    from crewai_tools import SerperDevTool
    print("[OK] SerperDevTool imported.")
except ImportError:
    print("[WARN] SerperDevTool not available.")
    SerperDevTool = None

In [ ]:
LLM_MODEL = "llama3.1:8b"
llm = ChatOllama(model=LLM_MODEL)

try:
    resp = llm.invoke("Reply OK")
    print("[OK] Ollama live:", resp.content[:30])
except Exception as e:
    print("[WARN] Ollama down. Start: ollama pull", LLM_MODEL)

## 2. FileReadTool -- reading file contents

`FileReadTool` reads the contents of a file and returns it as a string. The
agent can use this to inspect data files, config files, or source code.

We create a temporary file first, then demonstrate reading it.

In [ ]:
# Create a sample file for demonstration.
sample_dir = Path(tempfile.gettempdir()) / "crewai_tools_demo"
sample_dir.mkdir(exist_ok=True)

sample_file = sample_dir / "sample_data.txt"
sample_file.write_text(
    "Name,Score,Grade\n"
    "Alice,95,A\n"
    "Bob,82,B\n"
    "Charlie,91,A\n"
    "Diana,76,C\n"
)
print("Sample file created:", sample_file)

In [ ]:
if FileReadTool is not None:
    # Instantiate the tool with no arguments -- it reads any path the agent requests.
    file_reader = FileReadTool()

    # Standalone usage: call .run() with the file path.
    try:
        content = file_reader.run(str(sample_file))
        print("=== File Content ===")
        print(content)
    except Exception as e:
        print("[ERROR] FileReadTool.run() failed:", e)

    # You can also restrict the tool to a specific directory at construction time.
    restricted_reader = FileReadTool(directory=str(sample_dir))
    print("\nRestricted to directory:", restricted_reader.directory)
else:
    print("[skipped] FileReadTool not installed.")

## 3. FileWriterTool -- writing content to files

`FileWriterTool` writes a string to a file. The agent can use it to save
reports, data, or generated content.

In [ ]:
if FileWriterTool is not None:
    file_writer = FileWriterTool()

    output_file = sample_dir / "output_report.txt"
    try:
        result = file_writer.run(
            str(output_file),
            "AI Agent Report\n"
            "================\n"
            "Date: 2026-08-26\n"
            "Status: All systems operational.\n"
        )
        print("Write result:", result[:100] if isinstance(result, str) else result)

        # Verify by reading it back.
        print("File contents:", output_file.read_text())
    except Exception as e:
        print("[ERROR] FileWriterTool.run() failed:", e)
else:
    print("[skipped] FileWriterTool not installed.")

## 4. DirectoryReadTool -- listing directory contents

`DirectoryReadTool` lists the contents of a directory. Useful for agents
that need to discover what files are available before reading them.

In [ ]:
if DirectoryReadTool is not None:
    dir_reader = DirectoryReadTool()

    try:
        listing = dir_reader.run(str(sample_dir))
        print("=== Directory Listing ===")
        print(listing)
    except Exception as e:
        print("[ERROR] DirectoryReadTool.run() failed:", e)
else:
    print("[skipped] DirectoryReadTool not installed.")

## 5. DirectorySearchTool -- searching for files by pattern

`DirectorySearchTool` searches for files matching a glob pattern within a
directory tree. Useful for finding specific file types.

In [ ]:
if DirectorySearchTool is not None:
    dir_searcher = DirectorySearchTool()

    try:
        # Search for .txt files in the sample directory.
        results = dir_searcher.run(str(sample_dir), "*.txt")
        print("=== Search Results (*.txt) ===")
        print(results)
    except Exception as e:
        # Some versions of DirectorySearchTool have different signatures.
        print("[INFO] DirectorySearchTool signature may differ. Detail:", e)
else:
    print("[skipped] DirectorySearchTool not installed.")

## 6. ScrapeWebsiteTool -- web page scraping

`ScrapeWebsiteTool` fetches a URL and returns the page content as text.
Useful for agents that need to read documentation, news articles, or APIs.

> **Guarded:** all external calls are wrapped in try/except. If the network
> is unavailable the notebook still runs.

In [ ]:
if ScrapeWebsiteTool is not None:
    scraper = ScrapeWebsiteTool()

    try:
        # Scrape a simple public page.
        page_content = scraper.run("https://example.com")
        print("=== Scraped Content (first 300 chars) ===")
        print(str(page_content)[:300])
    except Exception as e:
        print("[WARN] ScrapeWebsiteTool failed (network issue):", e)
        print("       This is expected if you are offline or behind a firewall.")
else:
    print("[skipped] ScrapeWebsiteTool not installed.")

## 7. SerperDevTool -- web search (API key required)

`SerperDevTool` performs web searches using the Serper API (serper.dev).
It requires a `SERPER_API_KEY` environment variable.

We guard this heavily: if no key is set, the tool is skipped with a clear
message.

In [ ]:
if SerperDevTool is not None:
    serper_key = os.environ.get("SERPER_API_KEY", "")
    if serper_key:
        searcher = SerperDevTool()
        try:
            search_results = searcher.run("latest developments in CrewAI 2026")
            print("=== Serper Search Results (first 400 chars) ===")
            print(str(search_results)[:400])
        except Exception as e:
            print("[WARN] SerperDevTool failed:", e)
    else:
        print("[INFO] SERPER_API_KEY not set -- skipping SerperDevTool demo.")
        print("       To use: add SERPER_API_KEY=your_key to your .env file.")
        print("       Get a free key at https://serper.dev")
else:
    print("[skipped] SerperDevTool not installed.")

## 8. Assigning tools to agents

Agents receive tools via the `tools` parameter (a list). The agent can then
decide when and how to use each tool during its reasoning loop.

Below we assign `FileReadTool` and `DirectoryReadTool` to an agent, then
give it a task that requires reading files.

In [ ]:
if FileReadTool is not None and DirectoryReadTool is not None:
    file_agent = Agent(
        role="File Analyst",
        goal="Read and summarize file contents from the data directory.",
        backstory=(
            "You are a meticulous analyst who reads files carefully "
            "and produces accurate summaries."
        ),
        llm=llm,
        tools=[
            FileReadTool(),              # Agent can read files.
            DirectoryReadTool(),         # Agent can list directories.
        ],
        allow_delegation=False,
        verbose=True,
    )

    file_task = Task(
        description=(
            "Read all files in the data directory and provide a summary "
            f"of each file's contents. Directory: {sample_dir}"
        ),
        expected_output="A summary of each file found in the directory.",
        agent=file_agent,
    )

    file_crew = Crew(
        agents=[file_agent],
        tasks=[file_task],
        process=Process.sequential,
        verbose=False,
    )

    try:
        file_result = file_crew.kickoff()
        print("\n=== File Analysis ===")
        print(file_result.raw)
    except Exception as e:
        print("[demo skipped]", e)
else:
    print("[skipped] FileReadTool or DirectoryReadTool not available.")

## 9. Running tools standalone with `tool.run()`

Every CrewAI tool has a `.run()` method that accepts arguments and returns
a string result. This is useful for testing tools before assigning them to
agents, or for using tools outside of the agent framework entirely.

In [ ]:
# Demonstrate standalone tool usage.
if FileReadTool is not None:
    reader = FileReadTool()

    # Read the sample file.
    raw = reader.run(str(sample_file))
    print("=== Standalone FileReadTool.run() ===")
    print("Return type:", type(raw).__name__)
    print("Content preview:", str(raw)[:200])

    # Count lines.
    line_count = str(raw).count("\n")
    print("Line count:", line_count)

if FileWriterTool is not None:
    writer = FileWriterTool()

    standalone_output = sample_dir / "standalone_output.txt"
    writer.run(str(standalone_output), "Written by standalone FileWriterTool.run()")
    print("\nStandalone write complete:", standalone_output.read_text())

## 10. Tool error handling patterns

Tools can fail for many reasons: missing files, network errors, permission
issues, invalid API keys. Always wrap tool calls in try/except when building
production crews.

Below we demonstrate defensive tool usage patterns.

In [ ]:
# Pattern 1: Try/except around standalone tool.run().
if FileReadTool is not None:
    reader = FileReadTool()
    nonexistent = sample_dir / "does_not_exist.txt"

    try:
        content = reader.run(str(nonexistent))
        print("Content:", content)
    except Exception as e:
        print("[expected error] File not found:", e)

# Pattern 2: Pre-check before calling tool.run().
if FileWriterTool is not None:
    writer = FileWriterTool()
    safe_path = sample_dir / "safe_output.txt"

    # Ensure the parent directory exists before writing.
    safe_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        writer.run(str(safe_path), "Safe write succeeded.")
        print("Safe write:", safe_path.read_text())
    except Exception as e:
        print("[ERROR] Write failed:", e)

# Pattern 3: Validate tool output before using it.
if FileReadTool is not None:
    reader = FileReadTool()
    try:
        raw_output = reader.run(str(sample_file))
        # Check that output is non-empty and contains expected content.
        if raw_output and "Alice" in str(raw_output):
            print("\nValidation passed: file contains expected data.")
        else:
            print("\nValidation warning: unexpected file content.")
    except Exception as e:
        print("[ERROR] Read failed:", e)

## 11. Complete pattern: agent with tools + task + crew

In [ ]:
if FileReadTool is not None and FileWriterTool is not None:
    analyst_agent = Agent(
        role="Data Analyst",
        goal="Read data, analyze it, and save a report.",
        backstory="You are a thorough data analyst.",
        llm=llm,
        tools=[FileReadTool(), FileWriterTool()],
        allow_delegation=False,
        verbose=True,
    )

    analysis_task = Task(
        description=(
            f"Read the CSV file at {sample_file}, calculate the average "
            "score, and write a summary report to "
            f"{sample_dir / 'analysis_report.txt'}."
        ),
        expected_output=(
            "A report file written to disk with the average score "
            "and a brief analysis."
        ),
        agent=analyst_agent,
    )

    analysis_crew = Crew(
        agents=[analyst_agent],
        tasks=[analysis_task],
        process=Process.sequential,
        verbose=False,
    )

    try:
        analysis_result = analysis_crew.kickoff()
        print("\n=== Analysis Result ===")
        print(analysis_result.raw)
        # Check if the report was written.
        report_path = sample_dir / "analysis_report.txt"
        if report_path.exists():
            print("\n=== Report on Disk ===")
            print(report_path.read_text())
    except Exception as e:
        print("[demo skipped]", e)
else:
    print("[skipped] Required tools not available.")

## 12. Key takeaways

| Tool                 | Purpose                          | Requires API Key? |
|----------------------|----------------------------------|-------------------|
| `FileReadTool`       | Read file contents               | No                |
| `FileWriterTool`     | Write content to files           | No                |
| `DirectoryReadTool`  | List directory contents          | No                |
| `DirectorySearchTool`| Search files by glob pattern     | No                |
| `ScrapeWebsiteTool`  | Scrape web page content          | No                |
| `SerperDevTool`      | Web search via Serper API        | Yes               |

- All tools have a `.run()` method for standalone usage.
- Assign tools to agents via `Agent(tools=[...])`.
- Always wrap tool calls in try/except for production resilience.
- Next module: build a complete research assistant crew.